# 01 — Quality control

Everything the analysis set depends on is decided here, in code, so that the
Methods section can point at a rule rather than at a judgement call.

The notebook reads only:

* `data/derived/balance_data_2026.csv` — the wide analysis table (MATLAB output)
* `data/derived/VR-App_output.csv` — the per-trial log written by the same pipeline
* `data/derived/TrialOrder_Part2.xlsx` — participant list, body measures, file indices
* `data/raw/vr/` — the raw VR recordings, for duration and marker health

and writes two files:

* `data/derived/qc_trial_inventory.csv` — one row per raw recording
* `data/derived/qc_exclusions.csv` — what is dropped from the analysis set and why

In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DERIVED = REPO / "data" / "derived"
RAW_VR = REPO / "data" / "raw" / "vr"

# Acquisition constants, from the study protocol and verified against stim_pitch
SR = 90            # Hz, after resampling
T_TRIAL = 210      # s, full recording
T_QUIET = 50       # s, quiet stance before the visual perturbation starts
CYCLE = 20         # s, one stimulus cycle -> 8 cycles per trial, the first discarded

# QC thresholds — set here so the Methods text and the code cannot drift apart
MIN_DURATION = 209.0   # s, a recording shorter than this is dropped
MAX_ZERO_FRAC = 0.5    # a marker channel flat at zero for more than half the samples
MAX_CALIB_DRIFT = 0.10 # m, difference in mean HMD height between the two conditions

CONDITIONS = [f"{c}_t{t}" for c in ("HB", "LB") for t in (1, 2, 3, 4)]

order = pd.read_excel(DERIVED / "TrialOrder_Part2.xlsx")
PARTICIPANTS = order["Subject ID"].dropna().tolist()
print(f"{len(PARTICIPANTS)} participants x {len(CONDITIONS)} conditions "
      f"= {len(PARTICIPANTS) * len(CONDITIONS)} cells")

14 participants x 8 conditions = 112 cells


## 1. Coverage of the analysis table

How many participant x condition cells actually carry a value, and which do not.

In [2]:
balance = pd.read_csv(DERIVED / "balance_data_2026.csv")

coverage = (
    balance.set_index("ID")[[f"PeriodicPower {c}" for c in CONDITIONS]]
    .notna()
    .rename(columns=lambda c: c.replace("PeriodicPower ", ""))
)
empty = [(pid, cond) for pid, row in coverage.iterrows() for cond, ok in row.items() if not ok]

n_cells = coverage.size
print(f"cells: {n_cells} | filled: {n_cells - len(empty)} | empty: {len(empty)}")
for pid, cond in empty:
    print(f"  empty: {pid} {cond}")

cells: 112 | filled: 107 | empty: 5
  empty: DO11UB30 HB_t1
  empty: DO11UB30 HB_t2
  empty: DO11UB30 HB_t3
  empty: DO11UB30 HB_t4
  empty: DO11UB30 LB_t4


## 2. `rms_total` is not an independent measure

`run_VRApp_analysis_2026.m` line 75 builds the resultant sway from `com_ap`
twice instead of `com_ap` and `com_ml`:

```matlab
com_total = sqrt(cdiff(com_ap).^2 + cdiff(com_ap).^2) * sr;
```

If that is what happened, every `rms_total` must equal `sqrt(2) * rms_ap`
exactly. Checking rather than assuming.

In [3]:
ratio = pd.concat(
    [balance[f"rms_total {c}"] / balance[f"rms_ap {c}"] for c in CONDITIONS]
).dropna()

print(f"rms_total / rms_ap over {len(ratio)} cells: "
      f"min {ratio.min():.6f}, max {ratio.max():.6f}, sqrt(2) = {np.sqrt(2):.6f}")
print("identical to sqrt(2) everywhere:", bool(np.allclose(ratio, np.sqrt(2))))

rms_total / rms_ap over 107 cells: min 1.414214, max 1.414214, sqrt(2) = 1.414214
identical to sqrt(2) everywhere: True


The columns `rms_total *` carry no information beyond `rms_ap *` and are not used
further.

## 3. Repeatability of the trial log

`VR-App_output.csv` is appended to on every run of the pipeline, so a trial that
was processed more than once appears more than once. Before de-duplicating, it is
worth asking whether the repeated rows actually agree.

In [4]:
vr_log = pd.read_csv(DERIVED / "VR-App_output.csv")
by_trial = vr_log.groupby("fname")
measured = ["response sway power", "random sway power"]
fitted = ["visual Weight - W", "time delay - dt", "Loop Gain - Kp", "Kd",
          "Torque FB gain - Glp", "b", "sim Err"]

repeatability = pd.DataFrame({
    "trials with >1 value": [int((by_trial[c].nunique() > 1).sum()) for c in measured + fitted],
    "max spread": [
        by_trial[c].agg(
            lambda x: (x.max() - x.min()) / abs(x.mean()) if x.nunique() > 1 and x.mean() else np.nan
        ).max()
        for c in measured + fitted
    ],
}, index=measured + fitted)

print(f"{len(vr_log)} rows, {vr_log.fname.nunique()} unique trials, "
      f"repeat counts {sorted(map(int, vr_log.fname.value_counts().unique()))}\n")
print(repeatability.to_string(float_format=lambda v: f"{v:.1%}" if v == v else "—"))

387 rows, 107 unique trials, repeat counts [2, 3, 4, 6]

                      trials with >1 value  max spread
response sway power                      0         NaN
random sway power                        0         NaN
visual Weight - W                       79       16.0%
time delay - dt                         79        3.0%
Loop Gain - Kp                          79        1.3%
Kd                                      79        3.8%
Torque FB gain - Glp                    79      300.0%
b                                       79       12.6%
sim Err                                 78        0.1%


The two directly measured quantities are bit-identical across repeats. Every
parameter that comes out of the model fit is not: `pcl_ICfit_ml.m` optimises with
`GlobalSearch`, which is stochastic, so re-running the same trial returns a
slightly different optimum. The disagreement is usually negligible but reaches
16% for the visual weight and more for the weakly constrained torque-feedback
gain.

Two consequences, both of which the Methods section has to state:

1. `PeriodicPower` and `RemnantPower` are reproducible without qualification;
   model-derived parameters are reproducible only up to the optimiser.
2. De-duplication must be explicit. Repeated rows are collapsed by taking the
   median, which is stable for the fitted parameters and a no-op for the
   measured ones.

In [5]:
numeric = vr_log.select_dtypes("number").columns
trial_log = (
    vr_log.groupby("fname", as_index=False)
    .agg({"ID": "first", **{c: "median" for c in numeric}})
)
print(f"{len(vr_log)} rows -> {len(trial_log)} trials after collapsing repeats by median")

387 rows -> 107 trials after collapsing repeats by median


## 4. Raw trial inventory

For every raw recording of a main-sample participant: how long it actually is,
and whether the shoulder and hip markers were recorded at all. Duration is read
from the last line of the file, marker health from the first 30 s — neither needs
a full pass over 0.9 GB.

In [6]:
FOLDER = re.compile(r"^s(?P<pid>[A-Z0-9\u00c4\u00d6\u00dc]+)_(?P<session>[AB])_(?P<cond>H1?B|LB1?)_*$")


def final_timestamp(path: Path, nbytes: int = 4096) -> float:
    """Read the time column of the last row without reading the whole file."""
    with path.open("rb") as fh:
        fh.seek(max(0, path.stat().st_size - nbytes))
        tail = fh.read()
    last = tail.splitlines()[-1].decode("utf-8", "replace")
    return float(last.split(",", 1)[0])


records = []
for folder in sorted(RAW_VR.iterdir()):
    match = FOLDER.match(folder.name) if folder.is_dir() else None
    if not match or match["pid"] not in PARTICIPANTS:
        continue
    for path in sorted(folder.glob("*.csv")):
        head = pd.read_csv(path, nrows=30 * SR,
                           usecols=["time", "ypos", "shld_ypos", "hip_ypos"])
        records.append({
            "ID": match["pid"],
            "session": match["session"],
            "condition": "HB" if match["cond"].startswith("H") else "LB",
            "file": path.name,
            "bytes": path.stat().st_size,
            "duration_s": final_timestamp(path),
            "hmd_height_m": head.ypos.mean(),
            "shoulder_height_m": head.shld_ypos.mean(),
            "hip_height_m": head.hip_ypos.mean(),
            "shoulder_zero_frac": (head.shld_ypos == 0).mean(),
            "hip_zero_frac": (head.hip_ypos == 0).mean(),
        })

inventory = pd.DataFrame(records).sort_values(["ID", "condition", "file"]).reset_index(drop=True)
inventory.to_csv(DERIVED / "qc_trial_inventory.csv", index=False)
print(f"{len(inventory)} recordings from {inventory.ID.nunique()} participants "
      f"-> data/derived/qc_trial_inventory.csv")

117 recordings from 14 participants -> data/derived/qc_trial_inventory.csv


In [7]:
short = inventory[inventory.duration_s < MIN_DURATION]
print(f"shorter than {MIN_DURATION} s:")
print(short[["ID", "condition", "file", "duration_s"]].to_string(index=False))

flat = inventory[(inventory.shoulder_zero_frac > MAX_ZERO_FRAC)
                 | (inventory.hip_zero_frac > MAX_ZERO_FRAC)]
print("\nbody markers flat at zero:")
print(flat.groupby(["ID", "condition"])[["shoulder_zero_frac", "hip_zero_frac"]]
      .agg(["size", "mean"]).to_string())

shorter than 209.0 s:
      ID condition                                                       file  duration_s
BI20OE24        HB Screen_Balance and VR_1_sBI20OE24_B_H1B__t1_INCOMPLETE.csv  181.424744
BI20OE24        LB   Screen_Balance and VR_1_sBI20OE24_A_LB_t1_INCOMPLETE.csv  161.123047
CA04TU11        HB   Screen_Balance and VR_1_sCA04TU11_B_HB_t1_INCOMPLETE.csv   75.347960
PE16IN18        LB   Screen_Balance and VR_1_sPE16IN18_A_LB_t1_INCOMPLETE.csv  203.635376

body markers flat at zero:
                   shoulder_zero_frac      hip_zero_frac     
                                 size mean          size mean
ID       condition                                           
AN06AN18 HB                         5  0.0             5  1.0
         LB                         4  0.0             4  1.0
DO11UB30 HB                         4  1.0             4  1.0


Two distinct failures, and only one of them shows up as a gap in the analysis table.

`DO11UB30` has no shoulder and no hip signal in the high-boredom session, so the
centre of mass cannot be computed and those four trials are already empty above.
`AN06AN18` has a hip channel that is flat at zero in all nine recordings while the
head is tracked at 0.78 m for a participant of 1.57 m — the tracking origin is
wrong. The MATLAB code still produced numbers for this participant, because
`getCOM` falls back on the shoulder marker, but they do not describe body sway.

## 5. Calibration drift between conditions

The centre of mass is scaled by the measured marker heights, so a participant
whose VR floor calibration differs between the two sessions has a systematic
offset in exactly the contrast of interest.

In [8]:
calibration = (
    inventory.groupby(["ID", "condition"]).hmd_height_m.mean().unstack()
    .assign(drift_m=lambda d: (d.HB - d.LB).abs())
    .sort_values("drift_m", ascending=False)
)
print(calibration.round(3).to_string())

condition     HB     LB  drift_m
ID                              
EL30AD28   1.321  1.647    0.325
BI20OE24   1.484  1.493    0.009
CH10AL22   1.826  1.818    0.008
AN07IE11   1.667  1.661    0.006
AN23BE15   1.701  1.708    0.006
CH11RE22   1.567  1.561    0.006
CH08TU30   1.727  1.732    0.005
DO11UB30   1.749  1.753    0.004
RE13ON18   1.688  1.685    0.003
DI16UB31   1.517  1.521    0.003
AN06AN18   0.781  0.783    0.003
KA14RE15   1.608  1.607    0.001
PE16IN18   1.568  1.568    0.001
CA04TU11   1.681  1.682    0.000


One participant stands apart: for `EL30AD28` the head sits 33 cm lower in the
high-boredom session than in the low-boredom one, while everyone else agrees to
within a centimetre. That is a recalibration between sessions, not a change in
posture.

## 6. Exclusions

Each rule is applied in code and recorded with its reason, so the exclusion table
in the paper is generated rather than typed.

In [9]:
exclusions = []

for pid in inventory.loc[inventory.hip_zero_frac > MAX_ZERO_FRAC, "ID"].unique():
    trials = inventory[inventory.ID == pid]
    if trials.hip_zero_frac.gt(MAX_ZERO_FRAC).all():
        exclusions.append({"level": "participant", "ID": pid, "condition": "", "file": "",
                           "reason": "hip marker flat at zero in every recording; "
                                     "centre of mass not computable"})

for _, r in calibration[calibration.drift_m > MAX_CALIB_DRIFT].iterrows():
    exclusions.append({"level": "participant", "ID": r.name, "condition": "", "file": "",
                       "reason": f"VR calibration differs by {r.drift_m:.2f} m between "
                                 f"conditions; sway scaling not comparable"})

used_files = set(trial_log.fname)
for _, r in inventory[inventory.duration_s < MIN_DURATION].iterrows():
    if r.file in used_files:
        exclusions.append({"level": "trial", "ID": r.ID, "condition": r.condition,
                           "file": r.file,
                           "reason": f"recording ends at {r.duration_s:.1f} s of {T_TRIAL} s; "
                                     f"resampling would extrapolate"})

exclusions = pd.DataFrame(exclusions)
exclusions.to_csv(DERIVED / "qc_exclusions.csv", index=False)
print(exclusions.to_string(index=False))

      level       ID condition                                                       file                                                                           reason
participant AN06AN18                                                                             hip marker flat at zero in every recording; centre of mass not computable
participant EL30AD28                                                                      VR calibration differs by 0.33 m between conditions; sway scaling not comparable
      trial BI20OE24        HB Screen_Balance and VR_1_sBI20OE24_B_H1B__t1_INCOMPLETE.csv                 recording ends at 181.4 s of 210 s; resampling would extrapolate
      trial BI20OE24        LB   Screen_Balance and VR_1_sBI20OE24_A_LB_t1_INCOMPLETE.csv                 recording ends at 161.1 s of 210 s; resampling would extrapolate
      trial PE16IN18        LB   Screen_Balance and VR_1_sPE16IN18_A_LB_t1_INCOMPLETE.csv                 recording ends at 203.6 s of 210 s; res

In [10]:
dropped = set(exclusions.loc[exclusions.level == "participant", "ID"])
analysis_set = [p for p in PARTICIPANTS if p not in dropped]

paired = [p for p in analysis_set
          if coverage.loc[p, [f"HB_t{t}" for t in (1, 2, 3, 4)]].any()
          and coverage.loc[p, [f"LB_t{t}" for t in (1, 2, 3, 4)]].any()]

print(f"recruited                     {len(PARTICIPANTS)}")
print(f"after participant exclusions  {len(analysis_set)}  ({', '.join(sorted(dropped))} removed)")
print(f"with data in both conditions  {len(paired)}")
print(f"trial-level exclusions        {int((exclusions.level == 'trial').sum())}")

recruited                     14
after participant exclusions  12  (AN06AN18, EL30AD28 removed)
with data in both conditions  11
trial-level exclusions        3


## Summary

| check | result |
|---|---|
| coverage | 107 of 112 cells filled; the five gaps are one participant's high-boredom session plus one missing recording |
| `rms_total` | equals `sqrt(2) * rms_ap` in every cell — not an independent measure |
| repeatability | `PeriodicPower` and `RemnantPower` identical across repeated runs; fitted parameters differ by up to 16% |
| duration | three recordings that entered the pipeline end before 210 s |
| markers | one participant with no hip signal at all, one with no body markers in one session |
| calibration | one participant recalibrated between sessions by 33 cm |

Outputs: `data/derived/qc_trial_inventory.csv`, `data/derived/qc_exclusions.csv`.
Next: `02_prepare.ipynb` reshapes the wide table into one row per trial and joins
the boredom ratings.